In [1]:
import numpy as np
import pandas as pd

RAW = "../data/raw/speeds_synthetic.csv"            # .. goes up one level, out of notebooks/

df = pd.read_csv(RAW)                               # equivalent to MATLAB's readtable

print(df.shape)                                     # (rows, columns)
print(df.dtypes)                                    # data type of each column
df.head()                                           # first 5 rows

(104970, 3)
timestamp         str
speed_mph     float64
segment_id        str
dtype: object


,timestamp,speed_mph,segment_id
0,2023-02-28 08:25:00-05:00,59.237692,BQE-0231
1,2023-02-27 05:20:00-05:00,66.024130,BQE-0231
2,2023-05-14 20:45:00-04:00,65.665507,BQE-0231
3,2023-05-09 21:50:00-04:00,64.119572,BQE-0231
4,2023-03-21 16:25:00-04:00,56.255678,BQE-0231


In [2]:
df["timestamp"] = (
    pd.to_datetime(df["timestamp"], utc=True)       # utc=True normalizes any mixed offsets
    .dt.tz_convert("America/New_York")              # then shift to local wall-clock time
)
print(df["timestamp"].dtype)                        # expect datetime64[ns, America/New_York]

datetime64[us, America/New_York]


In [3]:
# 1. What's the coverage?
print(df["timestamp"].min(), "→", df["timestamp"].max())    # first and last timestamp

# 2. What's the sampling interval, really?
gaps = (
    df["timestamp"]
    .sort_values()                                  # chronological order
    .diff()                                         # difference between consecutive rows
    .value_counts()                                 # how often does each gap size occur
    .head(10)                                       # ten most common
)
print(gaps)

# 3. Are timestamps unique?
print("duplicate timestamps:", df["timestamp"].duplicated().sum())  # True flags summed = count

# 4. What does the distribution look like?
df["speed_mph"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])

2023-01-01 00:00:00-05:00 → 2023-12-31 23:55:00-05:00
timestamp
0 days 00:05:00    104719
0 days 00:10:00       200
0 days 00:00:00        50
Name: count, dtype: int64
duplicate timestamps: 50


count    104670.000000
mean         64.472450
std           6.306441
min           0.000000
1%           48.037268
5%           52.264603
25%          62.141217
50%          65.981475
75%          68.377708
95%          71.257005
99%          73.169664
max         255.000000
Name: speed_mph, dtype: float64